# Transformar  Datos Job Positions 
- Preprocesar la cadena json par corregir los problemas de calidad de datos 
- Transformar la cadena json en un objeto json
- Scribir los datos transformados en schema job_positons_json
- Acceder a elementos del objeto JSON 
- Eliminar los elementos duplicados dle array
- Anular el anidameinto de la array
- Escribir los datos en silver job_position

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

# 1. Leer la tabla origen (tal como está en tu captura)
df_job_positions = spark.table("zenviro.bronze.job_positions_py")

# 2. Definir el esquema del objeto JSON (agrega más StructField si hay otros campos en tu JSON) schema_of_json
schema_json = StructType([
    StructField("job_position_id", IntegerType(), True),
    StructField("title", StringType(), True),
    StructField("description", StringType(), True)
])

# 3. Convertir el texto JSON en estructura y agrupar en un array (si aún no es un array)
df_parsed = df_job_positions.withColumn("data", F.from_json(F.col("value"), schema_json))
df_array = df_parsed.agg(F.collect_list("data").alias("jobs_array"))

# 4. PASO 5: Eliminar elementos duplicados del array
df_sin_duplicados = df_array.withColumn("jobs_array", F.array_distinct(F.col("jobs_array")))

# 5. PASO 6: Anular el anidamiento del array y separar las columnas
df_transformado = df_sin_duplicados.select(F.explode(F.col("jobs_array")).alias("job")).select("job.*")

# Visualizar el resultado antes de guardar
display(df_transformado)

# 6. PASO 7: Escribir los datos transformados en el schema "silver" - "job_positions"
df_transformado.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("zenviro.silver.job_positions")